# Chapter 23: events, durable receipts and fresh status

Read README.md first. Predict archival and monitor counts from the ten opportunities before running. All times and quality flags are invented. This is one local process, not a measured network or clinical system.


In [ ]:
import json
from pathlib import Path
from tempfile import TemporaryDirectory
import distributed_platform as lab


## Inspect the schedule

Separate unique events from delivery attempts. Explain why a late event may belong in the archive while being unsuitable for the monitor.


In [ ]:
attempts = lab.schedule()
for now, principal, event in attempts:
    print(now, principal, event['seq'], event['source_ms'], event['quality_ok'])


In [ ]:
with TemporaryDirectory() as td:
    report = lab.run_fixture(Path(td)/'ledger.sqlite')
print(json.dumps({k:report[k] for k in ('expected_opportunities','archive_count','monitor_accepted','monitor_outcomes','restart','status_at_1000_ms')}, indent=2))
assert len(report['attempts']) == 12
assert report['archive_count'] == 9 and report['monitor_accepted'] == 7


## Reconstruct the evidence

Explain each duplicate, conflict, denied attempt, expiry and missing event. A last accepted record is historical evidence; after expiry its display status must be stale. Receiver restart does not recover the volatile monitor. Read the boundary tests, including the injected database insertion failure.


In [ ]:
import hashlib
from uuid import uuid4
folder = Path('outputs')/uuid4().hex
folder.mkdir(parents=True, exist_ok=False)
report['helper_sha256'] = hashlib.sha256(Path(lab.__file__).read_bytes()).hexdigest()
path = folder/'evidence.json'
path.write_text(json.dumps(report, indent=2, allow_nan=False), encoding='utf-8')
print(path, hashlib.sha256(path.read_bytes()).hexdigest())


## Extend one boundary

Add a bounded sender spool and test disconnection and a lost acknowledgement. Retain stable keys and original source times. Define overflow before testing. Keep those new observations separate from the baseline; no spool, broker, authenticated transport, concurrent delivery or process-crash experiment was supplied here. A digest compares bytes; it does not authenticate a publisher.
